In [1]:
import pandas as pd
import numpy as np
from collections import Counter

!pip install pandasql -q
from pandasql import sqldf
q = lambda sql: sqldf(sql, globals())

  Preparing metadata (setup.py) ... done


# PART 1: DATA CLEANING

In [2]:
df = pd.read_excel('/content/Case - Tarmac Technologies.xlsx', sheet_name='Data')

In [4]:
# 1.1 Parse flight-level time fields
# Format: "2025-02-28 - 17:40" → remove " - " → to datetime
for col in ['std', 'atd', 'sta', 'ata']:
    df[col + '_dt'] = pd.to_datetime(
        df[col].str.replace(' - ', ' ', regex=False), errors='coerce'
    )

# 1.2 Parse task-level time fields + cross-midnight fix
# Format: "14:43" pure time → combine with STD date
# Cross-midnight: if task hour - STD hour > 12 → task is previous day
# (affects 5 SFO red-eye flights with STD ~4AM but Pre-Flight at ~23:00)
df['base_date'] = df['std_dt'].dt.date
df['std_hour'] = df['std_dt'].dt.hour

def parse_task_time(row, time_col):
    val, base, std_h = row[time_col], row['base_date'], row['std_hour']
    if pd.isna(val) or pd.isna(base):
        return pd.NaT
    try:
        t = pd.to_datetime(str(val).strip(), format='%H:%M').time()
        dt = pd.Timestamp.combine(base, t)
        if t.hour - std_h > 12:
            dt -= pd.Timedelta(days=1)
        elif std_h - t.hour > 12:
            dt += pd.Timedelta(days=1)
        return dt
    except:
        return pd.NaT

for col in ['planning_start', 'actual_start', 'planning_end', 'actual_end']:
    df[col + '_dt'] = df.apply(lambda r: parse_task_time(r, col), axis=1)

# 1.3 Build turnaround-level summary table
# 5,564 rows → 52 turnarounds (one row per flight)
ta = df.groupby('turnaround_id').agg(
    airport=('airport_iata_code', 'first'),
    aircraft=('aircraft', 'first'),
    std=('std_dt', 'first'),
    atd=('atd_dt', 'first'),
    sta=('sta_dt', 'first'),
    ata=('ata_dt', 'first'),
    adc=('adc', 'first'),
    adct=('adct', 'first'),
).reset_index()

ta['dep_delay_min'] = (ta['atd'] - ta['std']).dt.total_seconds() / 60
ta['adc_diff_min'] = (ta['adc'] - ta['adct']).dt.total_seconds() / 60
ta['date'] = ta['std'].dt.date

# 1.4 Build task-level table
# Deduplicate: each turnaround × task = one row (remove custom_label level)
tasks = df[['turnaround_id', 'airport_iata_code', 'aircraft', 'task_name',
            'planning_start_dt', 'actual_start_dt',
            'planning_end_dt', 'actual_end_dt']].drop_duplicates(
    subset=['turnaround_id', 'task_name']
)
tasks['end_delay_min'] = (tasks['actual_end_dt'] - tasks['planning_end_dt']).dt.total_seconds() / 60
tasks['actual_duration_min'] = (tasks['actual_end_dt'] - tasks['actual_start_dt']).dt.total_seconds() / 60
tasks['planned_duration_min'] = (tasks['planning_end_dt'] - tasks['planning_start_dt']).dt.total_seconds() / 60

print(f"Cleaning done: {len(ta)} turnarounds, {len(tasks)} task records")

Cleaning done: 52 turnarounds, 1186 task records


# PART 2: DATA VALIDATION

In [6]:
# Check 1: ADC > ATD (logically impossible — doors closed after departure)
bad_adc = ta[ta.adc.notna() & ta.atd.notna() & (ta.adc > ta.atd)]
print(f"\nADC > ATD: {len(bad_adc)} flight(s)")
if len(bad_adc) > 0:
    print(bad_adc[['turnaround_id', 'airport', 'adc', 'atd']].to_string(index=False))

# Check 2: ADCT > STD (target close after scheduled departure)
adct_after_std = ta[ta.adct.notna() & (ta.adct > ta['std'])]
print(f"\nADCT > STD: {len(adct_after_std)} flight(s) — kept (likely updated targets)")
if len(adct_after_std) > 0:
    print(adct_after_std[['turnaround_id', 'airport', 'adct', 'std']].to_string(index=False))

# Check 3: actual_end < actual_start (task time flip)
time_flip = tasks[tasks.actual_start_dt.notna() & tasks.actual_end_dt.notna() & (tasks.actual_end_dt < tasks.actual_start_dt)]
print(f"\nactual_end < actual_start: {len(time_flip)} record(s)")
if len(time_flip) > 0:
    print(time_flip[['turnaround_id', 'task_name']].to_string(index=False))

# Fix: mark ADC > ATD as invalid (set ADC to NaT)
ta.loc[ta.adc.notna() & ta.atd.notna() & (ta.adc > ta.atd), 'adc'] = pd.NaT
ta.loc[ta.adc.isna(), 'adc_diff_min'] = np.nan

# Recalculate flags after fix
ta['otp_15'] = ta['dep_delay_min'] <= 15
ta['adc_on_time'] = ta['adc_diff_min'] <= 0

print(f"\nFix applied: 1 flight ADC set to NaT (ADC > ATD)")
print(f"Valid ADC flights: {ta.adc.notna().sum() & ta.adct.notna().sum()}")
print(f"OTP unaffected (STD/ATD all valid)")


ADC > ATD: 0 flight(s)

ADCT > STD: 2 flight(s) — kept (likely updated targets)
 turnaround_id airport                adct                 std
       1625392     SFO 2025-02-18 04:57:00 2025-02-18 03:55:00
       1631367     PPT 2025-02-20 17:22:00 2025-02-20 17:15:00

actual_end < actual_start: 1 record(s)
 turnaround_id    task_name
       1646816 Bag Delivery

Fix applied: 1 flight ADC set to NaT (ADC > ATD)
Valid ADC flights: 48
OTP unaffected (STD/ATD all valid)


# EDA

In [18]:
print(q("""
    SELECT
        COUNT(DISTINCT turnaround_id) AS turnarounds,
        COUNT(DISTINCT airport) AS airports,
        COUNT(DISTINCT aircraft) AS aircraft_types,
        MIN(date) AS date_from,
        MAX(date) AS date_to,
        COUNT(DISTINCT date) AS days,
        COUNT(DISTINCT task_name) AS task_types
    FROM (
        SELECT DISTINCT turnaround_id, airport, aircraft, date, task_name
        FROM ta CROSS JOIN (SELECT DISTINCT task_name FROM tasks)
    )
""").to_string(index=False))



 turnarounds  airports  aircraft_types  date_from    date_to  days  task_types
          52         3               2 2025-02-14 2025-02-28    15          39


# PART 3: KPI 1 — OTP (SQL)

In [7]:
print("\nOverall:")
print(q("""
    SELECT COUNT(*) AS total,
        SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) AS on_time,
        ROUND(SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS otp_pct
    FROM ta
""").to_string(index=False))

print("\nBy Airport:")
print(q("""
    SELECT airport, COUNT(*) AS total,
        SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) AS on_time,
        ROUND(SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS otp_pct
    FROM ta GROUP BY airport
""").to_string(index=False))

print("\nBy Aircraft:")
print(q("""
    SELECT aircraft, COUNT(*) AS total,
        SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) AS on_time,
        ROUND(SUM(CASE WHEN dep_delay_min <= 15 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS otp_pct
    FROM ta GROUP BY aircraft
""").to_string(index=False))


Overall:
 total  on_time  otp_pct
    52       40     76.9

By Airport:
airport  total  on_time  otp_pct
    ORY     39       30     76.9
    PPT      4        3     75.0
    SFO      9        7     77.8

By Aircraft:
aircraft  total  on_time  otp_pct
    A359     40       33     82.5
    A35K     12        7     58.3


# PART 4: KPI 2 — ADC COMPLIANCE (SQL)

In [8]:
print("\nOverall:")
print(q("""
    SELECT COUNT(*) AS total,
        SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) AS adc_on_time,
        ROUND(SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS adc_pct
    FROM ta
    WHERE adc_diff_min IS NOT NULL
""").to_string(index=False))

print("\nBy Airport:")
print(q("""
    SELECT airport, COUNT(*) AS total,
        SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) AS adc_on_time,
        ROUND(SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS adc_pct
    FROM ta WHERE adc_diff_min IS NOT NULL
    GROUP BY airport
""").to_string(index=False))

print("\nBy Aircraft:")
print(q("""
    SELECT aircraft, COUNT(*) AS total,
        SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) AS adc_on_time,
        ROUND(SUM(CASE WHEN adc_diff_min <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS adc_pct
    FROM ta WHERE adc_diff_min IS NOT NULL
    GROUP BY aircraft
""").to_string(index=False))


Overall:
 total  adc_on_time  adc_pct
    50           36     72.0

By Airport:
airport  total  adc_on_time  adc_pct
    ORY     37           26     70.3
    PPT      4            3     75.0
    SFO      9            7     77.8

By Aircraft:
aircraft  total  adc_on_time  adc_pct
    A359     39           30     76.9
    A35K     11            6     54.5


# PART 5: KPI 3a — DELAY RESPONSIBILITY (SQL)

In [9]:
print(q("""
    SELECT
        CASE
            WHEN adc_diff_min > 0 THEN 'Ground Ops'
            WHEN adc_diff_min <= 0 THEN 'ATC/Pushback'
            ELSE 'Unknown/Data Issue'
        END AS blame,
        COUNT(*) AS flights,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM ta WHERE dep_delay_min > 15), 1) AS pct
    FROM ta
    WHERE dep_delay_min > 15
    GROUP BY blame
    ORDER BY flights DESC
""").to_string(index=False))

             blame  flights  pct
        Ground Ops        9 75.0
      ATC/Pushback        2 16.7
Unknown/Data Issue        1  8.3


# PART 6: KPI 3b — ROOT CAUSE ANALYSIS (Python)

In [10]:
# Phase map: group 39 tasks into 4 phases by operational dependency
# Sub-phases (e.g. 1.1, 2.1) handle within-phase dependencies
phase_map = {
    # Phase 1: Arrival
    'ARR-TRC at Aircraft': 1.0, 'Disembark Pax': 1.0, 'Disembark Crew': 1.0,
    'PWD Arrival': 1.0, 'Offloading': 1.0,
    'Bag Delivery': 1.1, 'Cargo Delivery': 1.1, 'Decomp Panel': 1.1,
    # Phase 2: Ground services (parallel)
    'Cleaning': 2.0, 'Catering': 2.0, 'Water': 2.0, 'Security Check': 2.0,
    'Cargo transfer ramp': 2.0, 'Bag at Aircraft': 2.0, 'Check-In': 2.0,
    'Ticketing': 2.0, 'Transit Check-in': 2.0, 'Bag Search': 2.0,
    'Headset Agent': 2.0, 'Pre-Flight': 2.0, 'Briefing': 2.0, 'LIR': 2.0,
    'TRC-Pilots-brief': 2.0, 'Agent at Gate': 2.0, 'Sup Task 1': 2.0,
    'Cargo at Aircraft': 2.1,  # depends on Cargo transfer ramp
    # Phase 3: Departure preparation
    'FZFW': 3.0, 'Pre-Boarding': 3.0, 'PWD Departure': 3.0,
    'Boarding': 3.1, 'Loading': 3.1, 'NOTOC': 3.1,
    'Last Pax at Aircraft': 3.2, 'LDS': 3.2,
    'Flight File': 3.3, 'DEP-TRC at Aircraft': 3.3,
    # Phase 4: Door close & pushback
    'DEP-CDM': 4.0, 'Pushback Ready': 4.0, 'Tow out': 4.0,
}
tasks['phase'] = tasks['task_name'].map(phase_map)

def find_root_cause(tid):
    """For a delayed flight, scan phases in order.
    First task with end_delay > 5min = root cause.
    Later phases treated as cascading effects."""
    sub = tasks[(tasks.turnaround_id == tid) & (tasks.end_delay_min.notna())]
    sub = sub.sort_values(['phase', 'actual_end_dt'])
    for _, row in sub.iterrows():
        if row.end_delay_min > 5:
            return row.task_name, row.phase, row.end_delay_min
    return None, None, None

# Select ADC-delayed flights (not OTP-delayed)
# Reason: tasks influence door close time, not pushback time
delayed = ta[(ta.adc.notna()) & (ta.adct.notna()) & (ta.adc_on_time == False)].copy()
delayed['root_cause'], delayed['rc_phase'], delayed['rc_delay'] = zip(
    *delayed.turnaround_id.apply(find_root_cause)
)

print(f"\nADC-delayed flights: {len(delayed)}")

print(f"\nPer-flight root cause:")
for _, r in delayed.sort_values('adc_diff_min', ascending=False).iterrows():
    if r.root_cause:
        print(f"  {r.turnaround_id} | {r.airport} | {r.aircraft} | ADC{r.adc_diff_min:+.0f}min → Phase {r.rc_phase}: {r.root_cause} ({r.rc_delay:+.0f}min)")
    else:
        print(f"  {r.turnaround_id} | {r.airport} | {r.aircraft} | ADC{r.adc_diff_min:+.0f}min → No clear task delay")

print(f"\nOverall frequency:")
for task, count in Counter(delayed.root_cause.dropna()).most_common():
    print(f"  {task:<25s} {count} times ({count/len(delayed)*100:.0f}%)")

print(f"\nBy Airport:")
for apt in delayed.airport.unique():
    sub = delayed[delayed.airport == apt]
    counts = Counter(sub.root_cause.dropna())
    print(f"  {apt} ({len(sub)} delays): {', '.join([f'{t}({c})' for t, c in counts.most_common()])}")

print(f"\nBy Aircraft:")
for ac in delayed.aircraft.unique():
    sub = delayed[delayed.aircraft == ac]
    counts = Counter(sub.root_cause.dropna())
    print(f"  {ac} ({len(sub)} delays): {', '.join([f'{t}({c})' for t, c in counts.most_common()])}")


ADC-delayed flights: 14

Per-flight root cause:
  1617415 | ORY | A359 | ADC+78min → Phase 2.1: Cargo at Aircraft (+15min)
  1624598 | PPT | A359 | ADC+74min → Phase 1.1: Bag Delivery (+18min)
  1632656 | SFO | A359 | ADC+29min → Phase 2.0: Transit Check-in (+101min)
  1636933 | ORY | A359 | ADC+27min → Phase 2.0: TRC-Pilots-brief (+20min)
  1642239 | ORY | A359 | ADC+25min → Phase 3.1: Loading (+8min)
  1618416 | ORY | A35K | ADC+18min → Phase 1.0: Offloading (+25min)
  1643406 | ORY | A359 | ADC+18min → Phase 1.1: Bag Delivery (+22min)
  1637058 | ORY | A35K | ADC+18min → Phase 2.0: TRC-Pilots-brief (+21min)
  1639890 | ORY | A35K | ADC+13min → Phase 1.1: Bag Delivery (+8min)
  1624473 | ORY | A359 | ADC+10min → Phase 1.0: Disembark Pax (+17min)
  1636358 | ORY | A35K | ADC+3min → Phase 1.1: Bag Delivery (+25min)
  1622078 | ORY | A35K | ADC+3min → Phase 1.1: Bag Delivery (+11min)
  1650129 | ORY | A359 | ADC+2min → Phase 1.1: Bag Delivery (+15min)
  1623053 | SFO | A359 | ADC+1min 

# PART 7: ROOT CAUSE VALIDATION

In [11]:
#7.1 Sensitivity: does threshold choice affect conclusion? ---
print("\n7.1 Sensitivity Analysis:")
for threshold in [3, 5, 10]:
    rcs = []
    for tid in delayed.turnaround_id.values:
        sub = tasks[(tasks.turnaround_id == tid) & (tasks.end_delay_min.notna())].sort_values(['phase', 'actual_end_dt'])
        for _, row in sub.iterrows():
            if row.end_delay_min > threshold:
                rcs.append(row.task_name)
                break
    top = Counter(rcs).most_common(3)
    top_str = ', '.join([f"{t}({c})" for t, c in top])
    print(f"  Threshold {threshold}min: Top 3 = {top_str}")
print("  → Bag Delivery is #1 regardless of threshold")

#7.2 Independence: are the 6 Bag Delivery delays independent? ---
print("\n7.2 Independence Check:")
bag_tids = delayed[delayed.root_cause == 'Bag Delivery'].turnaround_id.values
bag_flights = ta[ta.turnaround_id.isin(bag_tids)].sort_values('date')
print(bag_flights[['turnaround_id', 'airport', 'aircraft', 'date']].to_string(index=False))
print(f"\n  {bag_flights.date.nunique()} different days, {bag_flights.airport.nunique()} airports, {bag_flights.aircraft.nunique()} aircraft types")
print("  → Independent events, not a one-off incident")

#7.3 Time pattern: clustered or spread out? ---
print("\n7.3 Time Pattern:")
delayed_with_date = delayed.merge(ta[['turnaround_id', 'date']], on='turnaround_id', suffixes=('', '_ta'))
daily = delayed_with_date.groupby('date').size()
for d, n in daily.items():
    bag_n = len(bag_flights[bag_flights.date == d])
    suffix = f" (incl. {bag_n} Bag Delivery)" if bag_n > 0 else ""
    print(f"  {d}: {n} delayed{suffix}")
print("  → Evenly distributed across 14-day window")


7.1 Sensitivity Analysis:
  Threshold 3min: Top 3 = Bag Delivery(5), Offloading(2), TRC-Pilots-brief(2)
  Threshold 5min: Top 3 = Bag Delivery(6), Offloading(2), TRC-Pilots-brief(2)
  Threshold 10min: Top 3 = Bag Delivery(5), Offloading(2), TRC-Pilots-brief(2)
  → Bag Delivery is #1 regardless of threshold

7.2 Independence Check:
 turnaround_id airport aircraft       date
       1622078     ORY     A35K 2025-02-16
       1624598     PPT     A359 2025-02-17
       1636358     ORY     A35K 2025-02-22
       1639890     ORY     A35K 2025-02-24
       1643406     ORY     A359 2025-02-25
       1650129     ORY     A359 2025-02-28

  6 different days, 2 airports, 2 aircraft types
  → Independent events, not a one-off incident

7.3 Time Pattern:
  2025-02-14: 1 delayed
  2025-02-15: 1 delayed
  2025-02-16: 1 delayed (incl. 1 Bag Delivery)
  2025-02-17: 3 delayed (incl. 1 Bag Delivery)
  2025-02-21: 1 delayed
  2025-02-22: 3 delayed (incl. 1 Bag Delivery)
  2025-02-24: 1 delayed (incl. 1 Bag

In [12]:
# EXPORT FOR TABLEAU
ta.to_csv('turnarounds.csv', index=False)

from google.colab import files
files.download('turnarounds.csv')



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>